In [1]:
!pip install easyocr
!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import string
import nltk
import spacy
import pandas as pd
from nltk.corpus import gutenberg, stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer

nltk.download('punkt')
nltk.download('gutenberg')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')
punctuations = set(string.punctuation)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [3]:
from spacy.lang.vi import Vietnamese

nlp = Vietnamese()
nlp.add_pipe('sentencizer')

In [8]:
import easyocr
import cv2
import json
import numpy as np
import regex

In [5]:
reader = easyocr.Reader(['vi']) # this needs to run only once to load the model into memory

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
# 📽 Input video
video_path = "/content/drive/MyDrive/Colab_resource/OCR_Video/VD_0001.mp4"

# 🎥 Open video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
fps = int(fps)
frame_data = {}
frame_num = 0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

cur_frame = 0
second_preFrame = 1
frame_milestone = second_preFrame * fps
print('fps: ', fps)
print('total frames: ', total_frames)

CP = 0

fps:  25
total frames:  26877


In [10]:
def sort_words_into_lines(results, y_tolerance=15):
    """
    results: list of dicts with keys: text, x, y, bbox
    y_tolerance: pixel threshold for grouping into the same line
    """
    lines = []

    for word in results:
        placed = False
        for line in lines:
            # Compare this word's baseline y to the mean baseline y of the current line
            mean_y = sum(w["avg_y"] for w in line) / len(line)
            if abs(word["avg_y"] - mean_y) <= y_tolerance:
                line.append(word)
                placed = True
                break
        if not placed:
            lines.append([word])

    # Sort lines top-to-bottom
    lines.sort(key=lambda line: sum(w["avg_y"] for w in line) / len(line))

    # Sort each line left-to-right
    for line in lines:
        line.sort(key=lambda w: w["x"])

    return lines

In [9]:
def merge_bounding_boxes(boxes):
    """Merge multiple bounding boxes into one."""
    min_x = min(b[0] for b in boxes)
    min_y = min(b[1] for b in boxes)
    max_x = max(b[0] + b[2] for b in boxes)
    max_y = max(b[1] + b[3] for b in boxes)

    return [min_x, min_y, max_x - min_x, max_y - min_y]

In [14]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if ((cur_frame / total_frames) * 100) > CP:
        print(f"Complete {CP}%")
        CP += 10

    # if cur_frame > 880:
    #   break

    cur_frame += 1
    if cur_frame % frame_milestone != 0:
        continue

    h, w, _ = frame.shape
    boxes = reader.readtext(frame, paragraph=False, width_ths=0.01)

    seconds = (cur_frame - 1) // fps + 1

    # 1️⃣ Collect per-word boxes and text
    results = []
    for bbox, text, conf in boxes:
        if conf > 0.7 and text.strip():
            x_coords = [point[0] for point in bbox]
            y_coords = [point[1] for point in bbox]

            x = min(x_coords)
            y = min(y_coords)
            bw = max(x_coords) - x
            bh = max(y_coords) - y

            results.append({
                "text": text.strip(),
                "x": x,
                "avg_y": (y + max(y_coords)) // 2,
                "bbox": [x / w, y / h, bw / w, bh / h]
            })

    lines = sort_words_into_lines(results, y_tolerance=15)

    ordered_words = [w for line in lines for w in line]  # flatten

    # Step 2: Build string and mapping of char positions
    text_str = ""
    mapping = []
    offset = 0
    for w in ordered_words:
        cleaned = regex.sub(r'[^\p{L}\s]', '', w["text"])
        cleaned = regex.sub(r'\s+', ' ', cleaned).strip()

        if cleaned:
            mapping.append({
                "bbox": w["bbox"],
                "start": offset,
                "end": offset + len(cleaned)
            })
            text_str += cleaned + " "
            offset += len(cleaned) + 1

    # if cur_frame >= 840 and cur_frame <=880:
    #     print("###############################################")
    #     print(text_str)

    # Step 3: Run spaCy
    doc = nlp(text_str)
    tokens_without_punct = [token for token in doc if not token.is_punct]

    # Step 4: Merge by spaCy sentence
    merged_data = []

    for token in tokens_without_punct:
        # if cur_frame >= 840 and cur_frame <=880:
        #   print(token.text.strip())
        # start position of the token in text_str
        token_start = token.idx
        token_end = token.idx + len(token.text)

        token_boxes = [
            m["bbox"] for m in mapping
            if m["start"] >= token_start and m["end"] <= token_end
        ]

        if token_boxes:
            merged_bbox = merge_bounding_boxes(token_boxes)
            merged_data.append({
                "bbox": merged_bbox,
                "text": token.text.strip()
            })

    # 6️⃣ Save result for this frame
    if merged_data:
        frame_data[str(seconds)] = merged_data

cap.release()

Complete 0%
Complete 10%
Complete 20%
Complete 30%
Complete 40%
Complete 50%
Complete 60%
Complete 70%
Complete 80%
Complete 90%


In [15]:
output = {
    "fps": fps,
    "data": frame_data
}

with open("ocr_data.json", "w") as f:
    json.dump(output, f, indent=2)

print("✅ OCR data saved to ocr_data.json")

✅ OCR data saved to ocr_data.json
